In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import plotly.express as px
import json

In [ ]:
# connect google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
newsdata_df = pd.read_csv("/content/drive/MyDrive/News_Analysis_Data/Thesis_Datasets_GH/newsdata_df_13257.csv")

In [ ]:
newsdata_df.head()

In [ ]:
newsdata_df.value_counts("language")

In [ ]:
# check the data range of the articles from pubDate column
newsdata_df["pubDate"].min(), newsdata_df["pubDate"].max()

### Load the IndicSBERT -> Generate Embeddings for articles' `title` for dataset from NewsData.io.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load the L3Cube-IndicSBERT model
model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli")
model = model.to(device)

In [ ]:
title_embeddings = []
with tqdm(total=len(newsdata_df), desc="Generating Title Embeddings", dynamic_ncols=True, leave=True) as pbar:
    for title in newsdata_df["title"].astype(str):
        embedding = model.encode(title, convert_to_tensor=True, device=device).cpu().numpy()  
        title_embeddings.append(embedding.tolist()) 
        pbar.update(1)

# Add embeddings to DataFrame
newsdata_df["title_embeddings"] = title_embeddings

In [ ]:
newsdata_df.head()

In [ ]:
# remove rows where language is vietnamese, khmer or swahili
newsdata_df = newsdata_df[~newsdata_df["language"].isin(["vietnamese", "khmer", "swahili"])]

In [ ]:
embedding_matrix = np.array(newsdata_df["title_embeddings"].tolist())

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=50,
    max_iter=2000,
    learning_rate=200,
    metric="cosine",
    random_state=42
)

tsne_result = tsne.fit_transform(embedding_matrix)


In [ ]:
# creating new DataFrame with selected columns and x/y
tsne_df = newsdata_df[["title", "source_name", "pubDate", "language"]].copy()
tsne_df["x"] = tsne_result[:, 0]
tsne_df["y"] = tsne_result[:, 1]

In [ ]:
# Plot t-SNE
fig = px.scatter(
    tsne_df, x="x", y="y", color="language",
    hover_data=["title", "language", "source_name"],
    title="t-SNE Clustering of News Articles"
)
fig.show()